In [4]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score


def fetch_and_prepare_dataset(data_path="creditcard.csv", test_ratio=0.2, seed=42):
    if not os.path.exists(data_path):
        target_dataset = "mlg-ulb/creditcardfraud"
        downloaded = False
        try:
            import kagglehub
            dataset_dir = kagglehub.dataset_download(target_dataset)
            csv_candidates = glob.glob(os.path.join(dataset_dir, "*.csv"))
            if csv_candidates:
                shutil.copy(csv_candidates[0], data_path)
                downloaded = True
        except Exception:
            pass

        if not downloaded:
            try:
                import kaggle
                kaggle.api.authenticate()
                kaggle.api.dataset_download_files(target_dataset, path=".", unzip=True)
                downloaded = True
            except Exception as err:
                raise FileNotFoundError(
                    f"Unable to automatically fetch {data_path} from Kaggle. "
                    f"Install kagglehub (`pip install kagglehub`) or configure ~/.kaggle/kaggle.json. Details: {err}"
                )

    raw_df = pd.read_csv(data_path).dropna()

    target_column = "Class"
    feature_columns = [col for col in raw_df.columns if col != target_column]

    features = raw_df[feature_columns].values
    labels = raw_df[target_column].values

    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        features, labels, test_size=test_ratio, random_state=seed, stratify=labels
    )

    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train_raw)
    X_test_norm = scaler.transform(X_test_raw)

    minority_train = X_train_norm[y_train == 1]
    majority_train = X_train_norm[y_train == 0]

    return X_train_norm, X_test_norm, y_train, y_test, minority_train, majority_train


class FraudDataGenerator(nn.Module):
    def __init__(self, latent_dim=32, feature_dim=30, hidden_units=128):
        super(FraudDataGenerator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(latent_dim, hidden_units),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(hidden_units, hidden_units),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(hidden_units, feature_dim)
        )

    def forward(self, noise):
        return self.network(noise)


class FraudDataDiscriminator(nn.Module):
    def __init__(self, feature_dim=30, hidden_units=128):
        super(FraudDataDiscriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(feature_dim, hidden_units),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_units, hidden_units // 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_units // 2, 1)
        )

    def forward(self, sample):
        return self.network(sample)


def train_generative_adversarial_model(minority_samples, latent_dim=32, epochs=60, batch_size=32, lr=0.0002, device="cpu"):
    feature_dim = minority_samples.shape[1]
    net_G = FraudDataGenerator(latent_dim=latent_dim, feature_dim=feature_dim).to(device)
    net_D = FraudDataDiscriminator(feature_dim=feature_dim).to(device)

    opt_G = optim.Adam(net_G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = optim.Adam(net_D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce_loss = nn.BCEWithLogitsLoss()

    fraud_dataset = TensorDataset(torch.tensor(minority_samples, dtype=torch.float32))
    train_loader = DataLoader(fraud_dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    history_g_loss = []
    history_d_loss = []

    for epoch in range(1, epochs + 1):
        total_d_step_loss = 0.0
        total_g_step_loss = 0.0
        steps = 0

        for real_batch, in train_loader:
            b_size = real_batch.size(0)
            real_batch = real_batch.to(device)
            real_target = torch.ones((b_size, 1), device=device)
            fake_target = torch.zeros((b_size, 1), device=device)

            opt_D.zero_grad()
            pred_real = net_D(real_batch)
            loss_d_real = bce_loss(pred_real, real_target)

            noise = torch.randn(b_size, latent_dim, device=device)
            fake_batch = net_G(noise)
            pred_fake = net_D(fake_batch.detach())
            loss_d_fake = bce_loss(pred_fake, fake_target)

            step_d_loss = loss_d_real + loss_d_fake
            step_d_loss.backward()
            opt_D.step()

            opt_G.zero_grad()
            pred_fake_for_g = net_D(fake_batch)
            step_g_loss = bce_loss(pred_fake_for_g, real_target)
            step_g_loss.backward()
            opt_G.step()

            total_d_step_loss += step_d_loss.item()
            total_g_step_loss += step_g_loss.item()
            steps += 1

        avg_epoch_g = total_g_step_loss / steps
        avg_epoch_d = total_d_step_loss / steps
        history_g_loss.append(avg_epoch_g)
        history_d_loss.append(avg_epoch_d)

        if epoch % 10 == 0 or epoch == epochs:
            print(f"Epoch [{epoch:03d}/{epochs:03d}] | Loss D: {avg_epoch_d:.4f} | Loss G: {avg_epoch_g:.4f}")

    return net_G, history_g_loss, history_d_loss


def synthesize_and_balance_data(generator_model, majority_data, minority_data, latent_dim=32, device="cpu"):
    num_samples_to_generate = len(majority_data) - len(minority_data)

    generator_model.eval()
    with torch.no_grad():
        latent_noise = torch.randn(num_samples_to_generate, latent_dim, device=device)
        generated_fraud = generator_model(latent_noise).cpu().numpy()

    X_balanced = np.vstack([majority_data, minority_data, generated_fraud])
    y_balanced = np.concatenate([
        np.zeros(len(majority_data)),
        np.ones(len(minority_data) + num_samples_to_generate)
    ])

    return X_balanced, y_balanced


def run_classification_benchmarks(X_train, y_train, X_test, y_test):
    classifiers = {
        "MLP Classifier": MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            max_iter=50,
            random_state=42,
            early_stopping=True
        ),
        "Linear SVM": LinearSVC(
            C=0.1,
            max_iter=2000,
            random_state=42
        )
    }

    results = {}

    for name, clf in classifiers.items():
        clf.fit(X_train, y_train)
        predictions = clf.predict(X_test)

        prec = precision_score(y_test, predictions, zero_division=0)
        rec = recall_score(y_test, predictions, zero_division=0)
        f1 = f1_score(y_test, predictions, zero_division=0)

        results[name] = {
            "model": clf,
            "predictions": predictions,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "confusion_matrix": confusion_matrix(y_test, predictions)
        }

        print(f"\n--- {name} ---")
        print(f"Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
        print(classification_report(y_test, predictions, target_names=["Legitimate", "Fraudulent"], digits=4))

    return results


def export_visualizations(g_losses, d_losses, benchmark_results):
    plt.figure(figsize=(8, 4))
    plt.plot(g_losses, label="Generator Loss")
    plt.plot(d_losses, label="Discriminator Loss")
    plt.title("GAN Training Convergence Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("convergence_plot.png", dpi=300)
    plt.close()

    for name, res in benchmark_results.items():
        cm = res["confusion_matrix"]
        safe_name = name.lower().replace(" ", "_")

        plt.figure(figsize=(5, 4))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Normal", "Fraud"],
            yticklabels=["Normal", "Fraud"]
        )
        plt.title(f"Confusion Matrix: {name}")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.savefig(f"confusion_matrix_{safe_name}.png", dpi=300)
        plt.close()


def main():
    runtime_device = "cuda" if torch.cuda.is_available() else "cpu"

    X_train, X_test, y_train, y_test, minority_data, majority_data = fetch_and_prepare_dataset(
        data_path="creditcard.csv", test_ratio=0.2, seed=42
    )

    generator, g_loss, d_loss = train_generative_adversarial_model(
        minority_samples=minority_data,
        latent_dim=32,
        epochs=60,
        batch_size=32,
        device=runtime_device
    )

    X_train_balanced, y_train_balanced = synthesize_and_balance_data(
        generator_model=generator,
        majority_data=majority_data,
        minority_data=minority_data,
        latent_dim=32,
        device=runtime_device
    )

    benchmarks = run_classification_benchmarks(
        X_train=X_train_balanced,
        y_train=y_train_balanced,
        X_test=X_test,
        y_test=y_test
    )

    export_visualizations(g_loss, d_loss, benchmarks)


if __name__ == "__main__":
    main()


Epoch [010/060] | Loss D: 1.0542 | Loss G: 0.6433
Epoch [020/060] | Loss D: 1.0311 | Loss G: 0.9115
Epoch [030/060] | Loss D: 0.8680 | Loss G: 1.0900
Epoch [040/060] | Loss D: 0.8168 | Loss G: 1.2453
Epoch [050/060] | Loss D: 1.0081 | Loss G: 1.2874
Epoch [060/060] | Loss D: 0.6938 | Loss G: 1.4035

--- MLP Classifier ---
Precision: 0.8041 | Recall: 0.7959 | F1: 0.8000
              precision    recall  f1-score   support

  Legitimate     0.9996    0.9997    0.9997     56864
  Fraudulent     0.8041    0.7959    0.8000        98

    accuracy                         0.9993     56962
   macro avg     0.9019    0.8978    0.8998     56962
weighted avg     0.9993    0.9993    0.9993     56962


--- Linear SVM ---
Precision: 0.8169 | Recall: 0.5918 | F1: 0.6864
              precision    recall  f1-score   support

  Legitimate     0.9993    0.9998    0.9995     56864
  Fraudulent     0.8169    0.5918    0.6864        98

    accuracy                         0.9991     56962
   macro avg   